In [ ]:
# Import the necessary modules for handling environment variables and for working with LangChain.
from dotenv import load_dotenv
from langchain_google_genai import ChatGoogleGenerativeAI

from langchain.tools import tool
from langchain.messages import HumanMessage

from pydantic import BaseModel
from typing import Literal

from langchain.agents import create_agent


load_dotenv()



True

In [ ]:
# We're going to use the gpt-3.5-turbo model from OpenAI, and we're setting the temperature to 0. 
# Temperature affects the randomness of the AI's responses. A temperature of 0 makes the output completely deterministic, 
# providing the same output every time for a given input.
chat = ChatOpenAI(model_name="gpt-3.5-turbo", temperature=0)

# Now we're going to create our chat templates. 
# These templates define the structure of the conversation, and we're going to use them to guide the AI's responses.

# The system message acts as the initial instruction for the AI. It sets the context for the conversation. 
# In our case, the AI is a helpful assistant that translates English to California surfer slang.
system_message_prompt = SystemMessagePromptTemplate.from_template("You are a helpful assistant that translates English to California surfer slang.")

# We then add an example of a human message and an AI message. These are just examples to guide the AI in the conversation.
example_human = HumanMessagePromptTemplate.from_template("Hi")
example_ai = AIMessagePromptTemplate.from_template("What's up, dude?")

# We also specify a template for future human messages. In this case, it's just the text of the message.
human_message_prompt = HumanMessagePromptTemplate.from_template("{text}")

# We then create a chat prompt from all these templates. The chat prompt is what we will use to guide the AI in the conversation.
chat_prompt = ChatPromptTemplate.from_messages([system_message_prompt, example_human, example_ai, human_message_prompt])

# We create a LangChain with our chat model and our chat prompt.
chain = LLMChain(llm=chat, prompt=chat_prompt)

# Finally, we run our chain with an example input, and print the result. In this case, the input is "I love programming."
# The AI will respond based on the templates we've given it and the input it receives.
print(chain.run("I love programming."))

In [ ]:
model = ChatGoogleGenerativeAI(
    # model="gemini-2.5-flash-lite",
    model="gemini-2.5-flash",
    temperature=1.0,  # Gemini 3.0+ defaults to 1.0
    max_tokens=None,
    timeout=None,
    max_retries=2,
    # other params...
)

messages = [
    (
        "system",
        "You are a helpful assistant that translates English to French. Translate the user sentence.",
    ),
    ("human", "I love programming."),
]
ai_msg = model.invoke(messages)
ai_msg

Both GOOGLE_API_KEY and GEMINI_API_KEY are set. Using GOOGLE_API_KEY.


In [19]:
# Define the tool
@tool(description="Get the current weather in a given location")
def get_weather(location: str) -> str:
    return "It's sunny."


# Initialize and bind (potentially multiple) tools to the model
model_with_tools = ChatGoogleGenerativeAI(model="gemini-2.5-flash").bind_tools([get_weather])

# Step 1: Model generates tool calls
messages = [HumanMessage("What's the weather in Boston?")]
ai_msg = model_with_tools.invoke(messages)
messages.append(ai_msg)

# Check the tool calls in the response
print(ai_msg.tool_calls)

# Step 2: Execute tools and collect results
for tool_call in ai_msg.tool_calls:
    # Execute the tool with the generated arguments
    tool_result = get_weather.invoke(tool_call)
    messages.append(tool_result)

# Step 3: Pass results back to model for final response
final_response = model_with_tools.invoke(messages)
final_response

Both GOOGLE_API_KEY and GEMINI_API_KEY are set. Using GOOGLE_API_KEY.


[{'name': 'get_weather', 'args': {'location': 'Boston'}, 'id': '804ad21c-132d-45fa-8699-d1f18f7004aa', 'type': 'tool_call'}]


AIMessage(content='The weather in Boston is sunny.', additional_kwargs={}, response_metadata={'finish_reason': 'STOP', 'model_name': 'gemini-2.5-flash', 'safety_ratings': [], 'model_provider': 'google_genai'}, id='lc_run--019b0743-9f31-7251-af0f-71aec26280a7-0', usage_metadata={'input_tokens': 83, 'output_tokens': 7, 'total_tokens': 90, 'input_token_details': {'cache_read': 0}})

In [14]:
def check_weather(location: str) -> str:
    '''Return the weather forecast for the specified location.'''
    return f"It's always sunny in {location}"


graph = create_agent(
    # model="gemini-2.5-flash-lite",
    model="gemini-2.5-flash",
    tools=[check_weather],
    system_prompt="You are a helpful assistant",
)

ImportError: Unable to import langchain_google_vertexai. Please install with `pip install -U langchain-google-vertexai`

In [ ]:




inputs = {"messages": [{"role": "user", "content": "what is the weather in sf"}]}
for chunk in graph.stream(inputs, stream_mode="updates"):
    print(chunk)

TypeError: "Could not resolve authentication method. Expected either api_key or auth_token to be set. Or for one of the `X-Api-Key` or `Authorization` headers to be explicitly omitted"